# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule:** rank every page by the percentile of its prior average search position (the average position from March 1-15). A worse (higher) prior position gets a higher priority score. This is the single signal validated in the capstone analysis - a richer blend and three separate machine learning models were all tested against it, and none beat it by a statistically significant margin, so the simplest version is used here.

**Reason codes this rule can output:**
- Visible in search but earning zero clicks in the prior window
- Inconsistent visibility - fewer than half of tracked days had impressions
- Already ranking below the corpus median position
- Ranking position is the primary driver - no secondary flags

In [5]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
base = "hf://datasets/FlyRank/internship-warehouse"

print("Connected.")

Connected.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
import pandas as pd
import numpy as np
import os

page_level = con.sql(f"""
    WITH first_half AS (
        SELECT client_hash_id, content_hash_id,
               AVG(gsc_impressions) as avg_impressions_h1,
               AVG(gsc_clicks) as avg_clicks_h1,
               AVG(gsc_avg_position) as avg_position_h1,
               COUNT(*) FILTER (WHERE gsc_impressions > 0) as active_days_h1
        FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE AND report_date <= '2026-03-15'
        GROUP BY client_hash_id, content_hash_id
    ),
    second_half AS (
        SELECT content_hash_id,
               AVG(gsc_avg_position) as avg_position_h2
        FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE AND report_date > '2026-03-15'
        GROUP BY content_hash_id
    )
    SELECT f.*, s.avg_position_h2
    FROM first_half f
    JOIN second_half s ON f.content_hash_id = s.content_hash_id
""").df()

print(f"Pages scored before cleaning: {len(page_level)}")

# Exclude avg_position_h1 = 0: not a real search position (positions start at 1).
# Found via Section 4's check: 868 pages had this value, and 100% of them sat in the
# Refresh Immediately tier purely because of it, not because of real decline signal.
before = len(page_level)
page_level = page_level[page_level["avg_position_h1"] > 0].copy()
print(f"Excluded {before - len(page_level)} pages with avg_position_h1 = 0 (data artifact, not a real ranking)")
print(f"Remaining: {len(page_level)}")

# Rank-normalized single-signal score: worse prior position -> higher priority.
# Percentile ranking is used instead of min/max scaling, since min/max was found in the
# capstone analysis to be badly distorted by a few extreme outlier values.
sorted_pos = np.sort(page_level["avg_position_h1"].values)
ranks = np.searchsorted(sorted_pos, page_level["avg_position_h1"].values, side="right") / len(sorted_pos)
page_level["baseline_score"] = 1 - ranks

page_level["percentile"] = page_level["baseline_score"].rank(pct=True)
page_level["priority_tier"] = np.select(
    [page_level["percentile"] >= 0.90, page_level["percentile"] >= 0.70],
    ["Refresh Immediately", "Monitor"], default="No Action"
)

median_position = page_level["avg_position_h1"].median()

def build_reasons(row):
    reasons = []
    if row["avg_clicks_h1"] == 0 and row["avg_impressions_h1"] > 0:
        reasons.append("Visible in search but earning zero clicks in the prior window")
    if row["active_days_h1"] < 8 and row["priority_tier"] != "No Action":
        reasons.append("Inconsistent visibility - fewer than half of tracked days had impressions")
    if row["avg_position_h1"] > median_position:
        reasons.append("Already ranking below the corpus median position")
    return " | ".join(reasons) if reasons else "Ranking position is the primary driver - no secondary flags"

page_level["reason_code"] = page_level.apply(build_reasons, axis=1)

# Tie-breaker added: baseline_score alone had a cluster of pages tied at the same
# percentile near the top-20 cutoff, so which pages showed up in ranks 18-20 was
# changing between runs. Sorting on content_hash_id as a secondary key makes the
# ranking reproducible - same top-20 every time, not just "for now."
ranked = page_level.sort_values(
    ["baseline_score", "content_hash_id"], ascending=[False, True]
).reset_index(drop=True)
ranked["rank"] = ranked.index + 1

output_cols = ["rank", "content_hash_id", "baseline_score", "percentile", "priority_tier", "reason_code",
               "avg_impressions_h1", "avg_clicks_h1", "avg_position_h1", "active_days_h1"]

os.makedirs("work/outputs", exist_ok=True)
ranked[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Written: work/outputs/baseline_action_score.csv ({len(ranked)} rows)")
print(f"\nTier distribution:\n{ranked['priority_tier'].value_counts()}")

Pages scored before cleaning: 141467
Excluded 868 pages with avg_position_h1 = 0 (data artifact, not a real ranking)
Remaining: 140599
Written: work/outputs/baseline_action_score.csv (140599 rows)

Tier distribution:
priority_tier
No Action              98419
Monitor                28120
Refresh Immediately    14060
Name: count, dtype: int64


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
top20 = ranked.head(20)[output_cols]
top20

,rank,content_hash_id,baseline_score,percentile,priority_tier,reason_code,avg_impressions_h1,avg_clicks_h1,avg_position_h1,active_days_h1
0,1,content_ec305a4a84cdb340,0.999993,1.000000,Refresh Immediately,Visible in search but earning zero clicks in t...,156.600000,0.0,0.012229,10
1,2,content_d89018d057259d5c,0.999986,0.999993,Refresh Immediately,Visible in search but earning zero clicks in t...,1.923077,0.0,0.015385,13
2,3,content_a6c3284707e06e0b,0.999979,0.999986,Refresh Immediately,Visible in search but earning zero clicks in t...,3.666667,0.0,0.020833,12
3,4,content_a18fb920bf78d8a0,0.999972,0.999979,Refresh Immediately,Visible in search but earning zero clicks in t...,9.615385,0.0,0.021457,13
4,5,content_675888a5cddb1690,0.999964,0.999972,Refresh Immediately,Visible in search but earning zero clicks in t...,4.466667,0.0,0.025000,15
5,6,content_e074834d3d00bc8e,0.999957,0.999964,Refresh Immediately,Visible in search but earning zero clicks in t...,2.000000,0.0,0.027778,9
6,7,content_b4ab2116222ec035,0.999950,0.999957,Refresh Immediately,Visible in search but earning zero clicks in t...,3.250000,0.0,0.035714,4
7,8,content_829c96c06870e291,0.999943,0.999950,Refresh Immediately,Visible in search but earning zero clicks in t...,2.928571,0.0,0.040816,14
8,9,content_53dde43c1721ecf2,0.999936,0.999943,Refresh Immediately,Visible in search but earning zero clicks in t...,3.250000,0.0,0.041667,4
9,10,content_4f902198232c55e4,0.999929,0.999936,Refresh Immediately,Visible in search but earning zero clicks in t...,2.500000,0.0,0.050000,4


Every page in this top 20 shares the same underlying pattern: it ranks in the top ~10% by
prior search position (baseline_score above 0.9998) but earns exactly zero clicks in the
prior window despite non-zero impressions. Confidence below is based on how much history
backs each row up (active_days_h1 and avg_impressions_h1), not copy-pasted.

1. content_ec305a4a84cdb340 - 156.6 avg daily impressions, 0 clicks, 10 active days.
   Confidence: High - large, consistent impression volume makes zero clicks hard to dismiss
   as chance. Wrong if: this query is satisfied by a featured snippet or knowledge panel,
   where ranking well earns visibility but no click is needed.
2. content_d89018d057259d5c - 13 active days, 1.9 avg impressions. Confidence: Medium - long
   history but very low traffic volume.
3. content_a6c3284707e06e0b - 12 active days, 3.7 avg impressions. Confidence: Medium, same
   reasons as above.
4. content_a18fb920bf78d8a0 - 13 active days, 9.6 avg impressions. Confidence: Medium-High -
   decent volume and long history together.
5. content_675888a5cddb1690 - 15 active days (full window), 4.5 avg impressions. Confidence:
   Medium-High - complete history supports this being a real pattern, not partial data.
6. content_e074834d3d00bc8e - 9 active days, 2.0 avg impressions. Confidence: Medium.
7. content_b4ab2116222ec035 - only 4 active days, 3.25 avg impressions. Confidence: Low -
   short history, could still be early noise rather than a settled pattern.
8. content_829c96c06870e291 - 14 active days, 2.9 avg impressions. Confidence: Medium-High.
9. content_53dde43c1721ecf2 - only 4 active days, 3.25 avg impressions. Confidence: Low, same
   reason as row 7.
10. content_4f902198232c55e4 - only 4 active days, 2.5 avg impressions. Confidence: Low, same
    reason as row 7.
11. content_1672aec89cbb2982 - 13 active days, 1.8 avg impressions. Confidence: Medium.
12. content_b00348592e2becad - 9 active days, 1.4 avg impressions (lowest volume in the top
    17). Confidence: Medium-Low - real history, but very little traffic to judge from.
13. content_5585f5c4793dac68 - 11 active days, 2.3 avg impressions. Confidence: Medium.
14. content_aba401583c6b8ff6 - 8 active days, 1.75 avg impressions. Confidence: Medium.
15. content_b0db647da3b6ba90 - 7 active days, 3.6 avg impressions. Confidence: Medium.
16. content_baf11dfb6980c0c2 - 13 active days, 26.5 avg impressions. Confidence: High -
    strong, consistent traffic makes the zero-click result stand out clearly.
17. content_f2a175ee4fa71a29 - 15 active days (full window), 119.3 avg impressions.
    Confidence: High - the strongest case in this list: excellent position, high and
    consistent traffic, and still zero clicks.
18. content_0baca1acd302c762 - only 5 active days, 2.0 avg impressions. Confidence: Low -
    short history.
19. content_1b326600ee2b3f5b - only 5 active days, 1.2 avg impressions (lowest volume in the
    top 20). Confidence: Low - short history and very little traffic to judge from.
20. content_2e365b64c875903e - only 5 active days, 1.4 avg impressions. Confidence: Low, same
    reason as row 18-19.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
# Leakage check: confirm no second-half/outcome column was used in the scoring itself
feature_cols_used = ["avg_impressions_h1", "avg_clicks_h1", "avg_position_h1", "active_days_h1"]
assert "avg_position_h2" not in feature_cols_used, "Leak: outcome column present in scoring features"
print("Leakage check passed: no second-half/outcome column used in the score.")

# Weak picks: pages flagged highly on very little data are the ones most likely to be wrong -
# a page seen only once or twice can look identical to a real decline purely by chance.
weak_picks = ranked[(ranked["priority_tier"] == "Refresh Immediately") & (ranked["active_days_h1"] <= 2)]
print(f"\nWeak picks (Refresh Immediately with 2 or fewer active days): {len(weak_picks)}")
weak_picks[output_cols].head(10)

Leakage check passed: no second-half/outcome column used in the score.

Weak picks (Refresh Immediately with 2 or fewer active days): 1159


,rank,content_hash_id,baseline_score,percentile,priority_tier,reason_code,avg_impressions_h1,avg_clicks_h1,avg_position_h1,active_days_h1
52,53,content_7f78cf8debf231bc,0.999623,0.999644,Refresh Immediately,Visible in search but earning zero clicks in t...,6.0,0.0,0.166667,1
79,80,content_abb4f77b0b31e9a2,0.999424,0.999435,Refresh Immediately,Visible in search but earning zero clicks in t...,4.5,0.0,0.214286,2
83,84,content_732fb7a87b6c0ddd,0.999403,0.999410,Refresh Immediately,Visible in search but earning zero clicks in t...,14.5,0.0,0.216346,2
84,85,content_0df1831dc2ef7938,0.999388,0.999399,Refresh Immediately,Visible in search but earning zero clicks in t...,9.0,0.0,0.222222,1
92,93,content_ef336707d572b7f3,0.999232,0.999296,Refresh Immediately,Visible in search but earning zero clicks in t...,1.5,0.0,0.250000,2
93,94,content_e26a9a158bac4dc6,0.999232,0.999296,Refresh Immediately,Visible in search but earning zero clicks in t...,1.5,0.0,0.250000,2
99,100,content_be14d527efc3400e,0.999232,0.999296,Refresh Immediately,Visible in search but earning zero clicks in t...,4.0,0.0,0.250000,1
101,102,content_eca1e07cce00d240,0.999232,0.999296,Refresh Immediately,Visible in search but earning zero clicks in t...,1.5,0.0,0.250000,2
106,107,content_4f3f4f50b6638dc9,0.999232,0.999296,Refresh Immediately,Visible in search but earning zero clicks in t...,1.5,0.0,0.250000,2
154,155,content_146f3cd2f2675113,0.998898,0.998905,Refresh Immediately,Visible in search but earning zero clicks in t...,33.0,0.0,0.323077,2


**Which picks look weak, and why:** pages flagged `Refresh Immediately` on very few active days (shown above) are the weakest picks - a page seen only once or twice can look like a decline purely by chance, not a real pattern. Name a specific row from the table above once you see it.

**Leakage confirmed:** no client names, product flags, or second-half/future-window columns (`avg_position_h2` or anything from March 16-31) were used anywhere in the scoring logic - confirmed by the assertion above.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.